# Обучение на витринах Olist

Готовые Parquet находятся в `marts/data`. Ноутбук загружает контракт признаков, делит данные по времени и обучает небольшой технический бейзлайн. Полноценный подбор моделей и оценка бизнес-качества — следующий этап.

Выберите Python из `.venv`. Меняйте `TASK`, чтобы использовать другую задачу.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'marts' / 'store.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from marts import MartStore

store = MartStore(ROOT / 'marts' / 'data')
print('Параметры:', store.manifest['parameters'])
print('Проверок пройдено:', store.manifest['validation']['checks_passed'])
print(pd.DataFrame([{'task': key, 'table': value['table'],
                     'rows': store.manifest['tables'][value['table']]['rows'],
                     'features': len(value['features'])}
                    for key, value in store.manifest['tasks'].items()]).to_string(index=False))

Параметры: {'train_from': '2017-01-01 00:00:00', 'train_to': '2018-08-15 00:00:00', 'sales_end': '2018-08-30 00:00:00', 'category_fit_to': '2017-04-03 00:00:00', 'smoothing_k': 10, 'active_window_days': 90, 'churn_window_days': 60, 'demand_from': '2017-01-02', 'snapshot_ts': '2018-10-29 12:27:35'}
Проверок пройдено: 21
               task                          table  rows  features
           delivery            ml_delivery_dataset 91267        61
          review_t0           ml_review_dataset_t0 95180       105
          review_ts           ml_review_dataset_ts 95180       128
       seller_churn        ml_seller_churn_dataset 18219        37
seller_segmentation ml_seller_segmentation_dataset  1224        37
             demand              ml_demand_dataset  7140        55


## Выбор задачи и признаков

Варианты: `delivery`, `review_t0`, `review_ts`, `seller_churn`, `demand`, `seller_segmentation`. `X` формируется по явному списку разрешённых признаков; поля меток и даты будущих событий туда не входят.

In [2]:
TASK = 'delivery'
task = store.task(TASK)
X, y = task.X, task.y
print('X:', X.shape, 'y:', None if y is None else y.shape)
print('Таргеты:', task.contract['targets'])
print(X.head(3).to_string())
print('\nНаибольшая доля пропусков:')
print(X.isna().mean().sort_values(ascending=False).head(10))

X: (91267, 61) y: (91267,)
Таргеты: ['delivery_days_from_approval']
  seller_state customer_state  n_zip_prefixes customer_region  is_customer_capital  dist_to_state_capital_km seller_region  distance_km  is_same_state  is_same_city  pair_n_delivered  pair_delivery_days_median  pair_delivery_days_smoothed  pair_last_delivery_days  pair_days_since_last_delivery  seller_state_n_delivered  seller_state_delivery_days_median  seller_state_delivery_days_smoothed  seller_state_transit_vs_route  seller_handling_days_mean  seller_handling_days_median  seller_handling_days_smoothed  seller_delivery_days_mean  seller_delivery_days_median  seller_delivery_days_smoothed  seller_n_orders_total  seller_n_shipped  seller_n_delivered  seller_age_days  seller_late_share  seller_late_share_smoothed  seller_orders_last_7d  seller_orders_last_30d  seller_backlog  city_n_delivered  city_transit_days_median  city_transit_days_smoothed  city_transit_vs_state  city_late_share_smoothed  city_transit_days_last_3

## Разбиение по времени

Препроцессинг обучается только на train. Из train и valid удаляются строки, чьи метки ещё не были известны на границе следующего периода. Для спроса это также предотвращает пересечение четырёхнедельных окон таргетов. Для сегментации используется последний доступный снимок без временного разбиения.

In [3]:
if TASK == 'seller_segmentation':
    parts = None
    print('Сегментация: последний снимок', task.frame.snapshot_date.max())
else:
    boundaries = ('2018-01-01', '2018-04-01') if TASK == 'seller_churn' else ('2018-04-01', '2018-06-01')
    parts = task.split(*boundaries)
    print(pd.DataFrame([{'part': name, 'rows': len(part.frame),
                        'from': part.frame[task.contract['time_column']].min(),
                        'to': part.frame[task.contract['time_column']].max()}
                       for name, part in parts.items()]).to_string(index=False))
    assert all(len(part.frame) for part in parts.values())

 part  rows                from                  to
train 59384 2017-01-05 11:56:06 2018-03-29 01:12:21
valid 10510 2018-04-01 00:11:32 2018-05-29 13:08:31
 test 15195 2018-06-01 00:39:55 2018-08-14 23:53:56


## Небольшой бейзлайн

Категории кодируются с обработкой новых значений; числовые пропуски — медианой train. Это проверка пригодности данных для обучения, без настройки гиперпараметров. Для доставки используется квантиль p50. Для спроса — отдельный регрессор на каждый горизонт. Кластеризация использует числовые показатели последнего снимка.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, roc_auc_score, average_precision_score
from threadpoolctl import threadpool_limits

# Limit native threads for a predictable local notebook runtime.
threadpool_limits(limits=2)
cat = task.contract['categorical_features']
num = [c for c in task.contract['features'] if c not in cat]
def prepared(frame):
    frame = frame.copy()
    for col in cat:
        frame[col] = frame[col].fillna('__missing__').astype(str)
    for col in num:
        frame[col] = pd.to_numeric(frame[col], errors='coerce').astype(float).replace([np.inf, -np.inf], np.nan)
    return frame

preprocessing = ColumnTransformer([
    ('numeric', SimpleImputer(strategy='median', keep_empty_features=True, add_indicator=True), num),
    ('category', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat),
])
if TASK == 'seller_segmentation':
    model = Pipeline([
        ('impute', SimpleImputer(strategy='median', keep_empty_features=True)),
        ('scale', StandardScaler()),
        ('cluster', KMeans(n_clusters=4, n_init=10, random_state=42)),
    ])
    cluster = model.fit_predict(prepared(X)[num])
    print(pd.Series(cluster).value_counts().sort_index().rename('sellers_per_cluster'))
else:
    classification = TASK in ['review_t0', 'review_ts', 'seller_churn']
    if classification:
        estimator = HistGradientBoostingClassifier(max_iter=40, max_leaf_nodes=15, random_state=42)
    elif TASK == 'delivery':
        estimator = HistGradientBoostingRegressor(loss='quantile', quantile=0.5, max_iter=40, max_leaf_nodes=15, random_state=42)
    else:
        estimator = MultiOutputRegressor(HistGradientBoostingRegressor(max_iter=40, max_leaf_nodes=15, random_state=42))
    model = Pipeline([('prepare', preprocessing), ('model', estimator)])
    model.fit(prepared(parts['train'].X), parts['train'].y)
    rows = []
    for name in ['valid', 'test']:
        part = parts[name]
        if classification:
            pred = model.predict_proba(prepared(part.X))[:, 1]
            rows.append({'part': name, 'ROC_AUC': roc_auc_score(part.y, pred),
                         'PR_AUC': average_precision_score(part.y, pred), 'positive_rate': float(part.y.mean())})
        else:
            pred = model.predict(prepared(part.X))
            row = {'part': name, 'MAE': mean_absolute_error(part.y, pred)}
            if TASK == 'demand':
                row['WAPE'] = float(np.abs(part.y.to_numpy() - pred).sum() / part.y.to_numpy().sum())
                # WAPE across all hierarchy levels counts an order at multiple levels.
            rows.append(row)
    print(pd.DataFrame(rows).to_string(index=False))

 part      MAE
valid 4.461542
 test 4.627506


## Другие витрины и замена источника

Контракт данных общий для локальных файлов и будущего SQL-источника. После реализации SQL-витрин с теми же столбцами достаточно передать `read_table`; код подготовки X и y и разбиения останется тем же. SQL-таблицы сейчас не требуются.

In [5]:
print(store.read('mart_seller_snapshot').head(2).to_string())
print('\nВсе таблицы:', ', '.join(sorted(store.manifest['tables'])))

# Будущая замена файлов на SQL:
# sql_store = MartStore(read_table=lambda name: pd.read_sql_table(name, engine, schema='ml_mart'))
# task = sql_store.task(TASK)


                          seller_id  tenure_days  recency_days  n_orders_90d  revenue_90d  n_orders_180d  revenue_180d  n_orders_lifetime  revenue_lifetime  revenue_growth_90d  revenue_share_platform_90d  revenue_rank_pct_90d  n_active_weeks_90d  max_gap_days_180d  orders_trend_slope_180d  n_products_180d  n_categories_180d     main_category  category_concentration_180d  avg_item_price_180d  freight_share_180d  n_customers_180d  n_customer_states_180d  avg_distance_km_180d  multi_seller_order_share_180d  handling_days_median_180d  late_share_180d  late_share_single_seller_180d  n_reviews_180d  bad_review_share_180d_smoothed  bad_review_share_on_time_180d_smoothed  bad_review_share_vs_category  mean_score_180d  cancel_share_180d seller_state seller_region  is_seller_capital  platform_orders_7d_vs_28d  is_low_data snapshot_date
0  0015a82c2db000af6aaaf3ae2ecb0532      4.07147      4.071470             1        895.0              1         895.0                  1             895.0       